# ♞ Neural Chess Lab
## A small, inspectable AlphaZero-style learner for Google Colab

This notebook is a **teaching system**, not a shortcut to a grandmaster engine. It contains the complete learning loop:

1. encode a chess position as tensors;
2. predict a policy (promising moves) and value (who is better);
3. improve the policy with Monte Carlo Tree Search (MCTS);
4. create training data by self-play;
5. update the network on its own games;
6. inspect and deliberately edit individual weights;
7. explore everything in a polished browser lab.

The default model is intentionally compact. A single Colab GPU can run it, students can read it, and short experiments finish during a class. Serious chess strength requires far more self-play, tuning, and compute.

### Learning goals

- Understand policy/value networks and residual blocks.
- See why legal-move masking matters.
- Trace PUCT selection and negamax value backup in MCTS.
- Compare raw neural-network choices with search-improved choices.
- Observe loss curves, replay-buffer growth, and parameter statistics.
- Change a weight, measure the effect, and restore a checkpoint.

> **Colab:** choose **Runtime → Change runtime type → T4 GPU**, then run cells from top to bottom.


In [ ]:
#@title 1 · Install the small dependencies
# Colab also ships packages such as Gradio and Google ADK that share FastAPI's
# Starlette dependency. These ranges keep the whole Colab environment compatible.
!pip -q install --upgrade "python-chess>=1.999,<2" "fastapi>=0.133,<1" "starlette>=1.3.1,<2" "uvicorn>=0.34,<1" "nest-asyncio>=1.6,<2"


In [ ]:
#@title 2 · Imports, reproducibility, and experiment settings
from __future__ import annotations

import copy
import io
import json
import math
import os
import random
import tempfile
import threading
import time
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import chess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LabConfig:
    # Model size. Changing these requires rebuilding the model.
    channels: int = 64
    residual_blocks: int = 4

    # Optimization.
    learning_rate: float = 2e-3
    weight_decay: float = 1e-4
    batch_size: int = 32
    replay_capacity: int = 15_000

    # Search and self-play.
    simulations: int = 24
    c_puct: float = 1.5
    dirichlet_alpha: float = 0.30
    dirichlet_fraction: float = 0.25
    temperature_moves: int = 20
    max_game_plies: int = 160

    # Reproducibility.
    seed: int = 7


CFG = LabConfig()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = CFG.seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
print(f"PyTorch {torch.__version__} | device={DEVICE}")
if DEVICE.type != "cuda":
    print("Tip: enable a GPU in Runtime → Change runtime type for much faster self-play.")


## 3 · Position and move representations

The network cannot consume a `chess.Board` directly. We encode it into **18 planes of size 8×8**:

| Planes | Meaning |
|---:|---|
| 0–5 | White pawn, knight, bishop, rook, queen, king |
| 6–11 | Black pawn, knight, bishop, rook, queen, king |
| 12 | Side to move (all ones for White) |
| 13–16 | White/Black king- and queen-side castling rights |
| 17 | Half-move clock, clipped and scaled |

Moves use the compact **8×8×73 AlphaZero-style action space**: 56 sliding directions/distances, 8 knight jumps, and 9 underpromotions. Queen promotions use an ordinary sliding plane. Only legal action indices are ever allowed during prediction or search.


In [ ]:
#@title 3 · Tensor encoder and the 4,672-action move map
INPUT_PLANES = 18
PLANES_PER_SQUARE = 73
ACTION_SIZE = 64 * PLANES_PER_SQUARE

PIECE_TO_PLANE = {
    (chess.WHITE, chess.PAWN): 0,
    (chess.WHITE, chess.KNIGHT): 1,
    (chess.WHITE, chess.BISHOP): 2,
    (chess.WHITE, chess.ROOK): 3,
    (chess.WHITE, chess.QUEEN): 4,
    (chess.WHITE, chess.KING): 5,
    (chess.BLACK, chess.PAWN): 6,
    (chess.BLACK, chess.KNIGHT): 7,
    (chess.BLACK, chess.BISHOP): 8,
    (chess.BLACK, chess.ROOK): 9,
    (chess.BLACK, chess.QUEEN): 10,
    (chess.BLACK, chess.KING): 11,
}

# Direction order is arbitrary but fixed. (file delta, rank delta)
RAY_DIRECTIONS = [(0, 1), (1, 1), (1, 0), (1, -1),
                  (0, -1), (-1, -1), (-1, 0), (-1, 1)]
KNIGHT_DELTAS = [(1, 2), (2, 1), (2, -1), (1, -2),
                 (-1, -2), (-2, -1), (-2, 1), (-1, 2)]
UNDERPROMOTIONS = [chess.KNIGHT, chess.BISHOP, chess.ROOK]


def encode_board(board: chess.Board) -> torch.Tensor:
    """Return a float32 [18, 8, 8] tensor in absolute board orientation."""
    x = torch.zeros((INPUT_PLANES, 8, 8), dtype=torch.float32)
    for square, piece in board.piece_map().items():
        rank = chess.square_rank(square)
        file = chess.square_file(square)
        x[PIECE_TO_PLANE[(piece.color, piece.piece_type)], rank, file] = 1.0
    x[12].fill_(1.0 if board.turn == chess.WHITE else 0.0)
    x[13].fill_(float(board.has_kingside_castling_rights(chess.WHITE)))
    x[14].fill_(float(board.has_queenside_castling_rights(chess.WHITE)))
    x[15].fill_(float(board.has_kingside_castling_rights(chess.BLACK)))
    x[16].fill_(float(board.has_queenside_castling_rights(chess.BLACK)))
    x[17].fill_(min(board.halfmove_clock, 100) / 100.0)
    return x


def _unit_and_distance(delta: int) -> Tuple[int, int]:
    if delta == 0:
        return 0, 0
    return (1 if delta > 0 else -1), abs(delta)


def move_to_index(move: chess.Move, board: chess.Board) -> int:
    """Map a legal move to one of 4,672 indices."""
    ff, fr = chess.square_file(move.from_square), chess.square_rank(move.from_square)
    tf, tr = chess.square_file(move.to_square), chess.square_rank(move.to_square)
    df, dr = tf - ff, tr - fr

    if move.promotion in UNDERPROMOTIONS:
        # Relative left/straight/right from the moving pawn's point of view.
        relative_df = df if board.turn == chess.WHITE else -df
        direction_slot = {-1: 0, 0: 1, 1: 2}[relative_df]
        piece_slot = UNDERPROMOTIONS.index(move.promotion)
        plane = 64 + piece_slot * 3 + direction_slot
    elif (df, dr) in KNIGHT_DELTAS:
        plane = 56 + KNIGHT_DELTAS.index((df, dr))
    else:
        uf, file_distance = _unit_and_distance(df)
        ur, rank_distance = _unit_and_distance(dr)
        distance = max(file_distance, rank_distance)
        direction = (uf, ur)
        if direction not in RAY_DIRECTIONS or not 1 <= distance <= 7:
            raise ValueError(f"Move {move.uci()} is not representable")
        plane = RAY_DIRECTIONS.index(direction) * 7 + (distance - 1)

    return move.from_square * PLANES_PER_SQUARE + plane


def legal_action_map(board: chess.Board) -> Dict[int, chess.Move]:
    return {move_to_index(move, board): move for move in board.legal_moves}


# Representation checks: ordinary move, castle, queen promotion, underpromotion.
for fen in [
    chess.STARTING_FEN,
    "r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1",
    "8/P7/8/8/8/8/7p/4K2k w - - 0 1",
]:
    b = chess.Board(fen)
    ids = list(legal_action_map(b))
    assert len(ids) == len(set(ids)), "Legal moves must map to unique actions"
    assert all(0 <= i < ACTION_SIZE for i in ids)

print("Encoded start position:", tuple(encode_board(chess.Board()).shape))
print("Action-space size:", ACTION_SIZE)


## 4 · The policy/value residual network

One shared convolutional trunk learns board features. Two heads then answer different questions:

- **Policy head:** which legal moves deserve attention?
- **Value head:** from the side-to-move perspective, is this position closer to a win (+1), draw (0), or loss (−1)?

Residual connections make deeper models easier to optimize. Batch normalization stabilizes activations. This model is deliberately much smaller than research-scale systems.


In [ ]:
#@title 4 · Residual policy/value network
class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class ChessNet(nn.Module):
    def __init__(self, channels: int = CFG.channels, blocks: int = CFG.residual_blocks):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(INPUT_PLANES, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        )
        self.tower = nn.Sequential(*[ResidualBlock(channels) for _ in range(blocks)])
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 2, 1, bias=False),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(2 * 8 * 8, ACTION_SIZE),
        )
        self.value_conv = nn.Sequential(
            nn.Conv2d(channels, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.value_fc = nn.Sequential(
            nn.Linear(8 * 8, channels),
            nn.ReLU(),
            nn.Linear(channels, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.tower(self.stem(x))
        policy_logits = self.policy_head(x)
        value = self.value_fc(self.value_conv(x)).squeeze(-1)
        return policy_logits, value


def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())


MODEL = ChessNet().to(DEVICE)
OPTIMIZER = torch.optim.AdamW(
    MODEL.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay
)
SCALER = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
MODEL_LOCK = threading.RLock()

print(MODEL)
print(f"Trainable parameters: {count_parameters(MODEL):,}")


In [ ]:
#@title 5 · Legal-move prediction helper
@torch.inference_mode()
def predict_position(board: chess.Board, model: Optional[nn.Module] = None):
    """Return {action: probability} over legal moves and a scalar value."""
    model = MODEL if model is None else model
    was_training = model.training
    model.eval()
    x = encode_board(board).unsqueeze(0).to(DEVICE)
    logits, value = model(x)
    legal = legal_action_map(board)
    if not legal:
        if was_training:
            model.train()
        return {}, terminal_value(board)
    indices = torch.tensor(list(legal), device=DEVICE)
    probs = torch.softmax(logits[0, indices], dim=0).float().cpu().numpy()
    if was_training:
        model.train()
    return {idx: float(p) for idx, p in zip(legal, probs)}, float(value.item())


def terminal_value(board: chess.Board) -> float:
    """Terminal result from the current side-to-move perspective."""
    if board.is_checkmate():
        return -1.0
    if board.is_game_over(claim_draw=True):
        return 0.0
    raise ValueError("terminal_value called on a non-terminal position")


board = chess.Board()
policy, value = predict_position(board)
top = sorted(policy.items(), key=lambda item: item[1], reverse=True)[:5]
print("Untrained value:", round(value, 4))
print("Top legal moves:", [(legal_action_map(board)[i].uci(), round(p, 4)) for i, p in top])
assert abs(sum(policy.values()) - 1.0) < 1e-5


## 5 · Monte Carlo Tree Search (MCTS)

The raw network is uncertain, especially before training. MCTS repeatedly simulates promising continuations:

1. **Select** children using `Q + U`: current value plus an exploration bonus.
2. **Expand** a new leaf using the policy head.
3. **Evaluate** that leaf using the value head (or the exact game result).
4. **Back up** the value, flipping its sign at every ply because the side to move alternates.

At the root, optional Dirichlet noise encourages self-play to discover different openings. The improved target policy is the normalized root visit count—not the network's original guess.


In [ ]:
#@title 6 · PUCT Monte Carlo Tree Search
class SearchNode:
    def __init__(self, prior: float = 0.0):
        self.prior = prior
        self.visit_count = 0
        self.value_sum = 0.0
        self.children: Dict[int, SearchNode] = {}

    @property
    def expanded(self) -> bool:
        return bool(self.children)

    @property
    def mean_value(self) -> float:
        return self.value_sum / self.visit_count if self.visit_count else 0.0


def expand_node(node: SearchNode, board: chess.Board) -> float:
    """Expand leaf and return its value from side-to-move perspective."""
    if board.is_game_over(claim_draw=True):
        return terminal_value(board)
    policy, value = predict_position(board)
    node.children = {action: SearchNode(prior) for action, prior in policy.items()}
    return value


def add_root_noise(node: SearchNode):
    actions = list(node.children)
    if not actions:
        return
    noise = np.random.dirichlet([CFG.dirichlet_alpha] * len(actions))
    frac = CFG.dirichlet_fraction
    for action, n in zip(actions, noise):
        child = node.children[action]
        child.prior = (1.0 - frac) * child.prior + frac * float(n)


def select_child(node: SearchNode) -> Tuple[int, SearchNode]:
    """PUCT. Child Q is negated because it is stored for the opponent."""
    best_score, best_action, best_child = -float("inf"), None, None
    root_scale = math.sqrt(node.visit_count + 1)
    for action, child in node.children.items():
        q = -child.mean_value
        u = CFG.c_puct * child.prior * root_scale / (child.visit_count + 1)
        score = q + u
        if score > best_score:
            best_score, best_action, best_child = score, action, child
    return best_action, best_child


def run_mcts(board: chess.Board, simulations: int = CFG.simulations,
             add_noise: bool = False) -> SearchNode:
    root = SearchNode()
    expand_node(root, board)
    if add_noise:
        add_root_noise(root)

    for _ in range(max(1, simulations)):
        scratch = board.copy(stack=False)
        node = root
        path = [node]

        while node.expanded and not scratch.is_game_over(claim_draw=True):
            action, node = select_child(node)
            move = legal_action_map(scratch)[action]
            scratch.push(move)
            path.append(node)

        value = expand_node(node, scratch)
        for visited in reversed(path):
            visited.visit_count += 1
            visited.value_sum += value
            value = -value
    return root


def root_policy(root: SearchNode, temperature: float = 1.0) -> Dict[int, float]:
    actions = list(root.children)
    if not actions:
        return {}
    visits = np.array([root.children[a].visit_count for a in actions], dtype=np.float64)
    if temperature <= 1e-3:
        probs = np.zeros_like(visits)
        probs[int(np.argmax(visits))] = 1.0
    else:
        adjusted = np.power(visits + 1e-10, 1.0 / temperature)
        probs = adjusted / adjusted.sum()
    return {a: float(p) for a, p in zip(actions, probs)}


def choose_with_mcts(board: chess.Board, simulations: int = CFG.simulations,
                     temperature: float = 0.0, add_noise: bool = False):
    root = run_mcts(board, simulations, add_noise)
    policy = root_policy(root, temperature)
    actions, probabilities = zip(*policy.items())
    action = int(np.random.choice(actions, p=probabilities))
    return legal_action_map(board)[action], policy, root


# Tiny search smoke test.
test_board = chess.Board()
move, improved, root = choose_with_mcts(test_board, simulations=4)
print("MCTS chose", move.uci(), "from", len(improved), "legal actions")


## 6 · Replay memory, self-play, and learning

Each stored training example is `(position, improved policy, final outcome)`. Policies are stored sparsely because only legal moves have non-zero targets. When sampled, they are expanded into a dense batch on the GPU.

The total objective is:

$$\mathcal{L} = -\pi^T \log p + (z-v)^2$$

where $\pi$ is the MCTS visit policy, $p$ the network policy, $z$ the eventual game outcome, and $v$ the predicted value. AdamW supplies weight decay. Mixed precision is enabled automatically on CUDA.


In [ ]:
#@title 7 · Replay buffer and self-play game generation
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.data = deque(maxlen=capacity)

    def __len__(self):
        return len(self.data)

    def add(self, state: torch.Tensor, sparse_policy: Dict[int, float], value: float):
        self.data.append((state.to(torch.uint8), sparse_policy, float(value)))

    def sample(self, batch_size: int):
        batch = random.sample(self.data, min(batch_size, len(self.data)))
        states = torch.stack([item[0].float() for item in batch]).to(DEVICE)
        policies = torch.zeros((len(batch), ACTION_SIZE), device=DEVICE)
        for row, (_, sparse, _) in enumerate(batch):
            idx = torch.tensor(list(sparse), dtype=torch.long, device=DEVICE)
            val = torch.tensor(list(sparse.values()), dtype=torch.float32, device=DEVICE)
            policies[row, idx] = val
        values = torch.tensor([item[2] for item in batch], dtype=torch.float32, device=DEVICE)
        return states, policies, values


REPLAY = ReplayBuffer(CFG.replay_capacity)
GAME_HISTORY = []
TRAIN_HISTORY = []


def play_self_game(simulations: int = CFG.simulations, verbose: bool = False):
    board = chess.Board()
    trajectory = []

    while not board.is_game_over(claim_draw=True) and board.ply() < CFG.max_game_plies:
        temperature = 1.0 if board.ply() < CFG.temperature_moves else 0.15
        move, policy, _ = choose_with_mcts(
            board, simulations=simulations, temperature=temperature, add_noise=True
        )
        trajectory.append((encode_board(board), policy, board.turn))
        board.push(move)

    outcome = board.outcome(claim_draw=True)
    winner = outcome.winner if outcome is not None else None  # cutoff is a draw
    for state, policy, side_to_move in trajectory:
        z = 0.0 if winner is None else (1.0 if side_to_move == winner else -1.0)
        REPLAY.add(state, policy, z)

    record = {
        "plies": len(trajectory),
        "result": board.result(claim_draw=True) if outcome else "1/2-1/2 (cutoff)",
        "termination": outcome.termination.name if outcome else "PLY_CUTOFF",
        "buffer": len(REPLAY),
    }
    GAME_HISTORY.append(record)
    if verbose:
        print(record)
    return record


def generate_self_play(games: int = 1, simulations: int = CFG.simulations,
                       progress=None):
    records = []
    for game_index in range(games):
        records.append(play_self_game(simulations, verbose=True))
        if progress:
            progress(game_index + 1, games, records[-1])
    return records


In [ ]:
#@title 8 · Gradient updates
def train_one_step():
    if len(REPLAY) == 0:
        raise RuntimeError("Replay buffer is empty. Generate a self-play game first.")
    MODEL.train()
    states, target_policy, target_value = REPLAY.sample(CFG.batch_size)
    OPTIMIZER.zero_grad(set_to_none=True)

    with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
        logits, value = MODEL(states)
        policy_loss = -(target_policy * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
        value_loss = F.mse_loss(value, target_value)
        loss = policy_loss + value_loss

    SCALER.scale(loss).backward()
    SCALER.unscale_(OPTIMIZER)
    grad_norm = torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 5.0)
    SCALER.step(OPTIMIZER)
    SCALER.update()

    with torch.no_grad():
        entropy = -(F.softmax(logits, 1) * F.log_softmax(logits, 1)).sum(1).mean()
    metrics = {
        "step": len(TRAIN_HISTORY) + 1,
        "loss": float(loss.detach().cpu()),
        "policy_loss": float(policy_loss.detach().cpu()),
        "value_loss": float(value_loss.detach().cpu()),
        "entropy": float(entropy.detach().cpu()),
        "grad_norm": float(torch.as_tensor(grad_norm).cpu()),
        "buffer": len(REPLAY),
    }
    TRAIN_HISTORY.append(metrics)
    return metrics


def train_steps(steps: int = 10, progress=None):
    metrics = []
    for i in range(steps):
        row = train_one_step()
        metrics.append(row)
        if progress:
            progress(i + 1, steps, row)
        if i == 0 or (i + 1) % 10 == 0:
            print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
    return metrics


### First controlled experiment

The following cells are intentionally **not run automatically**. Start with one low-search game, train for a few steps, then compare predictions. On a T4, increase simulations and games gradually.


In [ ]:
#@title 9 · Optional: generate self-play data { run: "auto" }
# Switch this on when you are ready. Keeping it off makes "Run all" fast.
RUN_SELF_PLAY = False  #@param {type:"boolean"}
GAMES = 1              #@param {type:"integer"}
SIMULATIONS = 12       #@param {type:"integer"}

if RUN_SELF_PLAY:
    with MODEL_LOCK:
        generate_self_play(games=GAMES, simulations=SIMULATIONS)
    print("Replay examples:", len(REPLAY))
else:
    print("Self-play skipped. Enable RUN_SELF_PLAY or use the browser lab.")


In [ ]:
#@title 10 · Optional: train on the replay data { run: "auto" }
RUN_TRAINING = False  #@param {type:"boolean"}
STEPS = 10            #@param {type:"integer"}

if RUN_TRAINING:
    with MODEL_LOCK:
        train_steps(STEPS)
else:
    print("Training skipped. Enable RUN_TRAINING after generating replay data, or use the browser lab.")


## 7 · Inspect, edit, save, and restore weights

Neural networks are just named tensors. The helpers below expose their shapes and statistics and can change one scalar. Editing weights is excellent for sensitivity experiments—but large values can destabilize the entire model, so save a checkpoint first.


In [ ]:
#@title 11 · Weight laboratory and checkpoints
def parameter_report(limit: Optional[int] = None):
    rows = []
    with torch.no_grad():
        for name, p in MODEL.named_parameters():
            x = p.detach().float()
            rows.append({
                "name": name,
                "shape": list(p.shape),
                "count": p.numel(),
                "mean": float(x.mean().cpu()),
                "std": float(x.std(unbiased=False).cpu()),
                "min": float(x.min().cpu()),
                "max": float(x.max().cpu()),
                "l2": float(x.norm().cpu()),
            })
    return rows[:limit] if limit else rows


def set_weight(parameter_name: str, index: Tuple[int, ...], value: float):
    params = dict(MODEL.named_parameters())
    if parameter_name not in params:
        raise KeyError(f"Unknown parameter: {parameter_name}")
    p = params[parameter_name]
    if len(index) != p.ndim:
        raise ValueError(f"Expected {p.ndim} indices for shape {tuple(p.shape)}")
    if any(i < 0 or i >= size for i, size in zip(index, p.shape)):
        raise IndexError(f"Index {index} is outside shape {tuple(p.shape)}")
    with torch.no_grad():
        old = float(p[index].item())
        p[index] = float(value)
    return old, float(p[index].item())


def save_checkpoint(path="chess_lab_checkpoint.pt"):
    torch.save({
        "model": MODEL.state_dict(),
        "optimizer": OPTIMIZER.state_dict(),
        "config": asdict(CFG),
        "train_history": TRAIN_HISTORY,
        "game_history": GAME_HISTORY,
    }, path)
    return path


def load_checkpoint(path="chess_lab_checkpoint.pt"):
    payload = torch.load(path, map_location=DEVICE)
    MODEL.load_state_dict(payload["model"])
    if "optimizer" in payload:
        OPTIMIZER.load_state_dict(payload["optimizer"])
    TRAIN_HISTORY[:] = payload.get("train_history", [])
    GAME_HISTORY[:] = payload.get("game_history", [])
    return payload


report = parameter_report(limit=6)
for row in report:
    print(row)

# Example (uncomment after saving a checkpoint):
# old, new = set_weight("stem.0.weight", (0, 0, 0, 0), 0.25)
# print("changed", old, "→", new)


## 8 · Browser lab

The next cell starts a FastAPI server inside the Colab runtime. Its HTML, CSS, and JavaScript are embedded here, so the notebook remains a single portable file.

Features:

- play against the current model;
- compare raw policy and MCTS recommendations;
- launch self-play + training without blocking the page;
- follow replay size and loss curves;
- inspect every named parameter tensor;
- edit an individual scalar weight;
- download a checkpoint.

The web app is intended for a private classroom runtime. Do not expose it as a permanent public service without adding authentication, request limits, and deployment hardening.


In [ ]:
#@title 12 · FastAPI service and elegant HTML/CSS/JS interface
from fastapi import FastAPI, HTTPException, Query
from fastapi.responses import HTMLResponse, JSONResponse, Response
from pydantic import BaseModel, Field
import uvicorn


APP_HTML = r"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width,initial-scale=1">
  <title>Neural Chess Lab</title>
  <style>
    :root{--ink:#17201e;--muted:#61706c;--paper:#f5f1e8;--panel:#fffdf7;--line:#d8d1c4;
      --teal:#116466;--teal2:#1f8a83;--gold:#d8a93f;--red:#b94b4b;--shadow:0 16px 48px #263b3520}
    *{box-sizing:border-box} body{margin:0;color:var(--ink);font:15px/1.5 Inter,ui-sans-serif,system-ui;
      background:radial-gradient(circle at 8% 0,#dfeee8 0,transparent 32%),var(--paper)}
    header{padding:28px clamp(18px,5vw,64px) 14px;display:flex;align-items:end;justify-content:space-between;gap:18px}
    h1{font-family:Georgia,serif;font-size:clamp(30px,5vw,54px);font-weight:500;line-height:1;margin:0}
    h1 span{color:var(--teal)} .eyebrow{text-transform:uppercase;letter-spacing:.15em;font-size:11px;color:var(--teal);font-weight:800}
    .status{display:flex;align-items:center;gap:8px;color:var(--muted)} .dot{width:9px;height:9px;background:#7cbe62;border-radius:50%;box-shadow:0 0 0 4px #7cbe6225}
    main{padding:12px clamp(18px,5vw,64px) 54px;max-width:1500px;margin:auto}
    nav{display:flex;gap:8px;overflow:auto;border-bottom:1px solid var(--line);margin-bottom:22px}
    nav button{background:none;border:0;padding:12px 16px;color:var(--muted);font-weight:700;cursor:pointer;border-bottom:3px solid transparent}
    nav button.active{color:var(--teal);border-color:var(--teal)}
    .tab{display:none}.tab.active{display:block}.grid{display:grid;grid-template-columns:minmax(320px,590px) minmax(300px,1fr);gap:22px}
    .card{background:#fffdf9;border:1px solid #ded7cb;border-radius:18px;padding:20px;box-shadow:var(--shadow)}
    .card h2,.card h3{font-family:Georgia,serif;font-weight:500;margin:0 0 12px}.card h2{font-size:26px}
    .board{width:min(100%,560px);aspect-ratio:1;display:grid;grid-template-columns:repeat(8,1fr);overflow:hidden;border-radius:12px;box-shadow:0 8px 28px #14262230}
    .sq{border:0;padding:0;display:grid;place-items:center;font-size:clamp(28px,5vw,52px);cursor:pointer;position:relative;font-family:"Arial Unicode MS",serif}
    .light{background:#e8dbc1}.dark{background:#69918a}.sq.selected{box-shadow:inset 0 0 0 5px var(--gold)}
    .sq.target:after{content:"";width:22%;height:22%;border-radius:50%;background:#d5a62dbf;position:absolute}
    .sq.last{box-shadow:inset 0 0 0 5px #fff7}.toolbar{display:flex;flex-wrap:wrap;gap:9px;margin-top:14px}
    button,.button{border:0;border-radius:9px;padding:10px 14px;font:inherit;font-weight:750;cursor:pointer;background:var(--teal);color:white}
    button.secondary,.button.secondary{background:#e7ece8;color:var(--ink)}button.danger{background:var(--red)}button:disabled{opacity:.5;cursor:wait}
    input,select{width:100%;padding:10px 11px;border:1px solid var(--line);border-radius:9px;background:white;font:inherit;color:var(--ink)}
    label{display:block;font-size:12px;font-weight:800;color:var(--muted);margin-bottom:5px;text-transform:uppercase;letter-spacing:.05em}
    .fields{display:grid;grid-template-columns:repeat(auto-fit,minmax(130px,1fr));gap:12px;margin:14px 0}
    .metrics{display:grid;grid-template-columns:repeat(auto-fit,minmax(115px,1fr));gap:10px;margin-bottom:18px}
    .metric{padding:14px;background:#edf3ef;border-radius:12px}.metric b{display:block;font-size:23px;font-family:Georgia,serif}.metric span{color:var(--muted);font-size:12px}
    table{width:100%;border-collapse:collapse;font-size:13px}th,td{text-align:left;border-bottom:1px solid var(--line);padding:9px 7px}th{color:var(--muted);font-size:11px;text-transform:uppercase}
    .scroll{overflow:auto;max-height:520px}.bar{height:9px;background:#e1e5e0;border-radius:9px;overflow:hidden}.bar i{display:block;height:100%;background:linear-gradient(90deg,var(--teal),var(--gold));width:0}
    .move-row{display:grid;grid-template-columns:64px 1fr 62px;gap:8px;align-items:center;margin:9px 0}.mini{height:7px;background:#e4e5df;border-radius:5px;overflow:hidden}.mini i{height:100%;display:block;background:var(--teal2)}
    canvas{width:100%;height:230px;background:#fbfaf4;border-radius:12px}.note{padding:12px 14px;background:#fff5d8;border-left:4px solid var(--gold);border-radius:7px;color:#66552b}
    .lesson{max-width:850px}.lesson h2{margin-top:28px}.lesson code{background:#e7ece8;padding:2px 5px;border-radius:4px}.muted{color:var(--muted)}
    .toast{position:fixed;right:24px;bottom:24px;padding:12px 16px;border-radius:10px;color:white;background:var(--ink);opacity:0;transform:translateY(10px);transition:.2s;pointer-events:none}.toast.show{opacity:1;transform:none}
    @media(max-width:850px){.grid{grid-template-columns:1fr}header{align-items:start;flex-direction:column}.sq{font-size:10vw}}
  </style>
</head>
<body>
<header><div><div class="eyebrow">Self-learning systems · interactive studio</div><h1>Neural <span>Chess</span> Lab</h1></div><div class="status"><i class="dot"></i><span id="device">connecting…</span></div></header>
<main>
  <nav><button class="active" data-tab="play">Play & Analyze</button><button data-tab="train">Self-Train</button><button data-tab="weights">Weights</button><button data-tab="learn">Learn</button></nav>

  <section id="play" class="tab active"><div class="grid">
    <div class="card"><div id="board" class="board"></div><div class="toolbar"><button id="resetGame">New game</button><button id="flip" class="secondary">Flip board</button><button id="aiMove" class="secondary">Let model move</button></div><p id="gameText" class="muted">Select a piece, then a destination.</p></div>
    <div class="card"><h2>Position microscope</h2><label>FEN</label><input id="fen"><div class="fields"><div><label>Search simulations</label><input id="analysisSims" type="number" value="24" min="0" max="512"></div><div><label>Top moves</label><input id="topK" type="number" value="8" min="1" max="30"></div></div><button id="analyze">Analyze position</button><div class="metrics" style="margin-top:18px"><div class="metric"><b id="value">—</b><span>side-to-move value</span></div><div class="metric"><b id="legalCount">—</b><span>legal moves</span></div></div><div id="moves"></div><p class="note">Set simulations to 0 to see the raw network. Increase it to see how search changes the policy.</p></div>
  </div></section>

  <section id="train" class="tab"><div class="grid">
    <div class="card"><h2>Self-play workshop</h2><p class="muted">Generate games with the current model, add every position to replay memory, then learn from those examples.</p><div class="fields"><div><label>Games</label><input id="games" type="number" value="1" min="1" max="20"></div><div><label>Simulations / move</label><input id="sims" type="number" value="16" min="1" max="256"></div><div><label>Gradient steps</label><input id="steps" type="number" value="10" min="0" max="1000"></div></div><button id="startTraining">Start self-train cycle</button><div style="margin-top:18px"><div class="bar"><i id="progress"></i></div><p id="jobText" class="muted">Ready.</p></div><div class="toolbar"><a id="download" class="button secondary" href="api/checkpoint">Download checkpoint</a><button id="resetModel" class="danger">Reset model</button></div></div>
    <div class="card"><h2>Learning telemetry</h2><div class="metrics"><div class="metric"><b id="buffer">0</b><span>replay positions</span></div><div class="metric"><b id="trainStep">0</b><span>gradient steps</span></div><div class="metric"><b id="gamesPlayed">0</b><span>self-play games</span></div><div class="metric"><b id="loss">—</b><span>latest loss</span></div></div><canvas id="chart" width="720" height="230"></canvas><p class="muted">Total loss (teal), policy loss (gold), value loss (red). Curves are most useful over dozens of updates.</p></div>
  </div></section>

  <section id="weights" class="tab"><div class="grid">
    <div class="card"><h2>Parameter tensors</h2><p class="muted">Statistics are recomputed from the live model.</p><button id="refreshWeights">Refresh report</button><div class="scroll"><table><thead><tr><th>Name</th><th>Shape</th><th>Mean</th><th>Std</th><th>L2</th></tr></thead><tbody id="weightRows"></tbody></table></div></div>
    <div class="card"><h2>Edit one scalar</h2><p class="note">Download a checkpoint first. Extreme edits may create NaNs or erase learned behavior.</p><div class="fields" style="grid-template-columns:1fr"><div><label>Exact parameter name</label><input id="paramName" placeholder="stem.0.weight"></div><div><label>Index (comma-separated)</label><input id="paramIndex" placeholder="0,0,0,0"></div><div><label>New value</label><input id="paramValue" type="number" step="any" value="0.25"></div></div><button id="editWeight">Apply edit</button><p id="editResult" class="muted"></p><h3>Experiment protocol</h3><ol><li>Analyze a fixed FEN and record its value/top moves.</li><li>Save a checkpoint.</li><li>Change exactly one scalar or one small tensor region.</li><li>Analyze the same FEN again.</li><li>Explain why a local parameter edit can have a global effect.</li></ol></div>
  </div></section>

  <section id="learn" class="tab"><article class="card lesson"><h2>How the loop teaches itself</h2><p>The network starts with random weights. MCTS turns those weak guesses into a better move distribution by looking ahead. Self-play supplies positions and final outcomes. Gradient descent then makes the network imitate the search policy and predict the outcome. The improved network guides the next search more effectively, closing the loop.</p><h2>What to vary</h2><table><tr><th>Knob</th><th>Expected effect</th><th>Cost / risk</th></tr><tr><td>Simulations</td><td>Stronger policy targets</td><td>Nearly linear time cost</td></tr><tr><td>Residual blocks</td><td>More representational depth</td><td>Rebuild model; slower</td></tr><tr><td>Temperature moves</td><td>More opening diversity</td><td>Noisier games</td></tr><tr><td>Learning rate</td><td>Faster parameter movement</td><td>Instability if too high</td></tr><tr><td>Replay capacity</td><td>Retains more old experience</td><td>RAM use; stale data</td></tr></table><h2>Questions for students</h2><ol><li>Why is the value backed up with alternating signs?</li><li>Why train on MCTS visits instead of the raw policy?</li><li>What happens if Dirichlet noise is removed?</li><li>Why can loss decrease even when chess strength does not improve?</li><li>How would you evaluate a new checkpoint fairly?</li></ol><h2>Honest limitations</h2><p>This lab has no distributed actors, opening book, symmetry augmentation, transposition table, resignation calibration, or arena-based checkpoint gating. Those are excellent extension projects. A classroom run demonstrates the algorithm; it does not reproduce the compute scale of AlphaZero.</p></article></section>
</main><div id="toast" class="toast"></div>
<script>
const $=s=>document.querySelector(s), $$=s=>[...document.querySelectorAll(s)];
const START="rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1";
const glyph={K:'♔',Q:'♕',R:'♖',B:'♗',N:'♘',P:'♙',k:'♚',q:'♛',r:'♜',b:'♝',n:'♞',p:'♟'};
let fen=START, legal=[], selected=null, flipped=false, lastMove=null, busy=false;
function toast(s){let t=$('#toast');t.textContent=s;t.classList.add('show');setTimeout(()=>t.classList.remove('show'),2200)}
async function api(path,opt={}){let r=await fetch(path,{headers:{'Content-Type':'application/json'},...opt});if(!r.ok){let x=await r.json().catch(()=>({detail:r.statusText}));throw Error(x.detail||r.statusText)}return r.json()}
function fenMap(f){let out={},rank=7,file=0;for(let c of f.split(' ')[0]){if(c=='/'){rank--;file=0}else if(/\d/.test(c)){file+=+c}else{out['abcdefgh'[file]+(rank+1)]=c;file++}}return out}
function render(){let b=$('#board'), pieces=fenMap(fen), files=flipped?'hgfedcba':'abcdefgh',ranks=flipped?[1,2,3,4,5,6,7,8]:[8,7,6,5,4,3,2,1];b.innerHTML='';for(let rank of ranks)for(let file of files){let sq=file+rank,el=document.createElement('button');el.className='sq '+(((file.charCodeAt(0)-97+rank)%2)?'light':'dark');if(sq===selected)el.classList.add('selected');if(selected&&legal.some(m=>m.slice(0,2)===selected&&m.slice(2,4)===sq))el.classList.add('target');if(lastMove&&lastMove.includes(sq))el.classList.add('last');el.textContent=glyph[pieces[sq]]||'';el.onclick=()=>squareClick(sq);b.appendChild(el)}}
async function squareClick(sq){if(busy)return;if(selected){let choices=legal.filter(m=>m.slice(0,2)===selected&&m.slice(2,4)===sq);if(choices.length){let move=choices.find(m=>m.endsWith('q'))||choices[0];selected=null;await playMove(move);return}}let own=legal.some(m=>m.slice(0,2)===sq);selected=own?sq:null;render()}
async function analyze(updateBoard=true){try{let body={fen:$('#fen').value||fen,simulations:+$('#analysisSims').value,top_k:+$('#topK').value};let d=await api('api/analyze',{method:'POST',body:JSON.stringify(body)});if(updateBoard){fen=d.fen;legal=d.legal;$('#fen').value=fen;render()}$('#value').textContent=(d.value>=0?'+':'')+d.value.toFixed(3);$('#legalCount').textContent=d.legal.length;$('#moves').innerHTML=d.moves.map(m=>`<div class="move-row"><b>${m.san}</b><div class="mini"><i style="width:${Math.max(2,m.probability*100)}%"></i></div><span>${(m.probability*100).toFixed(1)}%</span></div>`).join('')}catch(e){toast(e.message)}}
async function playMove(move){busy=true;try{let d=await api('api/play',{method:'POST',body:JSON.stringify({fen,move,simulations:+$('#analysisSims').value})});fen=d.fen;legal=d.legal;lastMove=[move.slice(0,2),move.slice(2,4)];$('#fen').value=fen;$('#gameText').textContent=d.message;render();await analyze(false)}catch(e){toast(e.message)}finally{busy=false}}
async function letModelMove(){busy=true;try{let d=await api('api/model-move',{method:'POST',body:JSON.stringify({fen,simulations:+$('#analysisSims').value})});fen=d.fen;legal=d.legal;lastMove=[d.move.slice(0,2),d.move.slice(2,4)];$('#fen').value=fen;$('#gameText').textContent=d.message;render();await analyze(false)}catch(e){toast(e.message)}finally{busy=false}}
function drawChart(rows){let c=$('#chart'),x=c.getContext('2d'),W=c.width,H=c.height;x.clearRect(0,0,W,H);x.strokeStyle='#d8d1c4';for(let i=1;i<5;i++){x.beginPath();x.moveTo(35,i*H/5);x.lineTo(W-10,i*H/5);x.stroke()}if(!rows.length){x.fillStyle='#61706c';x.fillText('Train to populate this chart',35,H/2);return}let max=Math.max(...rows.flatMap(r=>[r.loss,r.policy_loss,r.value_loss]),.001);[['loss','#116466'],['policy_loss','#d8a93f'],['value_loss','#b94b4b']].forEach(([key,color])=>{x.strokeStyle=color;x.lineWidth=3;x.beginPath();rows.forEach((r,i)=>{let px=35+i*(W-50)/Math.max(1,rows.length-1),py=H-20-r[key]*(H-35)/max;i?x.lineTo(px,py):x.moveTo(px,py)});x.stroke()})}
async function poll(){try{let s=await api('api/status');$('#device').textContent=`${s.device.toUpperCase()} · ${s.parameters.toLocaleString()} parameters`;$('#buffer').textContent=s.replay;$('#trainStep').textContent=s.train_steps;$('#gamesPlayed').textContent=s.games;$('#loss').textContent=s.latest_loss==null?'—':s.latest_loss.toFixed(3);let j=s.job;$('#jobText').textContent=j.message||j.state;$('#progress').style.width=((j.progress||0)*100)+'%';$('#startTraining').disabled=j.state==='running';drawChart(s.history)}catch(e){$('#device').textContent='disconnected'}setTimeout(poll,1200)}
async function weights(){try{let rows=await api('api/weights');$('#weightRows').innerHTML=rows.map(r=>`<tr data-name="${r.name}"><td><b>${r.name}</b></td><td>${r.shape.join('×')}</td><td>${r.mean.toExponential(2)}</td><td>${r.std.toExponential(2)}</td><td>${r.l2.toFixed(2)}</td></tr>`).join('');$$('#weightRows tr').forEach(tr=>tr.onclick=()=>{$('#paramName').value=tr.dataset.name;toast('Parameter selected')})}catch(e){toast(e.message)}}
$$('nav button').forEach(b=>b.onclick=()=>{$$('nav button,.tab').forEach(x=>x.classList.remove('active'));b.classList.add('active');$('#'+b.dataset.tab).classList.add('active');if(b.dataset.tab==='weights')weights()});
$('#resetGame').onclick=()=>{fen=START;legal=[];selected=null;lastMove=null;$('#fen').value=fen;$('#gameText').textContent='New game. You play the side to move.';analyze()};$('#flip').onclick=()=>{flipped=!flipped;render()};$('#analyze').onclick=()=>analyze();$('#aiMove').onclick=()=>letModelMove();
$('#startTraining').onclick=async()=>{try{await api('api/self-train',{method:'POST',body:JSON.stringify({games:+$('#games').value,simulations:+$('#sims').value,steps:+$('#steps').value})});toast('Self-train cycle started')}catch(e){toast(e.message)}};
$('#refreshWeights').onclick=weights;$('#editWeight').onclick=async()=>{try{let d=await api('api/weights/edit',{method:'POST',body:JSON.stringify({name:$('#paramName').value,index:$('#paramIndex').value.split(',').map(x=>+x.trim()),value:+$('#paramValue').value})});$('#editResult').textContent=`Changed ${d.old.toExponential(5)} → ${d.new.toExponential(5)}`;toast('Weight changed');weights()}catch(e){toast(e.message)}};
$('#resetModel').onclick=async()=>{if(confirm('Reset all learned weights and clear replay/history?')){try{await api('api/reset',{method:'POST'});toast('Model reset')}catch(e){toast(e.message)}}};
$('#fen').value=fen;render();analyze();poll();
</script></body></html>
"""


class AnalyzeRequest(BaseModel):
    fen: str = chess.STARTING_FEN
    simulations: int = Field(default=0, ge=0, le=512)
    top_k: int = Field(default=8, ge=1, le=30)


class PlayRequest(BaseModel):
    fen: str
    move: Optional[str] = None
    simulations: int = Field(default=24, ge=0, le=512)


class TrainRequest(BaseModel):
    games: int = Field(default=1, ge=1, le=20)
    simulations: int = Field(default=16, ge=1, le=256)
    steps: int = Field(default=10, ge=0, le=1000)


class WeightEdit(BaseModel):
    name: str
    index: List[int]
    value: float


app = FastAPI(title="Neural Chess Lab", docs_url="/docs")
JOB_LOCK = threading.Lock()
JOB = {"state": "idle", "progress": 0.0, "message": "Ready."}


def parse_board(fen: str) -> chess.Board:
    try:
        return chess.Board(fen)
    except ValueError as exc:
        raise HTTPException(400, f"Invalid FEN: {exc}")


def board_payload(board: chess.Board):
    outcome = board.outcome(claim_draw=True)
    return {
        "fen": board.fen(),
        "legal": [m.uci() for m in board.legal_moves],
        "game_over": outcome is not None,
        "result": board.result(claim_draw=True) if outcome else None,
    }


def model_move(board: chess.Board, simulations: int):
    if board.is_game_over(claim_draw=True):
        raise HTTPException(400, "The game is already over")
    if simulations == 0:
        policy, _ = predict_position(board)
        action = max(policy, key=policy.get)
        return legal_action_map(board)[action]
    return choose_with_mcts(board, simulations=simulations, temperature=0.0)[0]


@app.get("/", response_class=HTMLResponse)
def home():
    return APP_HTML


@app.get("/api/status")
def status():
    return {
        "device": str(DEVICE),
        "parameters": count_parameters(MODEL),
        "replay": len(REPLAY),
        "train_steps": len(TRAIN_HISTORY),
        "games": len(GAME_HISTORY),
        "latest_loss": TRAIN_HISTORY[-1]["loss"] if TRAIN_HISTORY else None,
        "history": TRAIN_HISTORY[-120:],
        "job": dict(JOB),
    }


@app.post("/api/analyze")
def analyze(req: AnalyzeRequest):
    board = parse_board(req.fen)
    if board.is_game_over(claim_draw=True):
        value, distribution = terminal_value(board), {}
    elif req.simulations:
        root = run_mcts(board, req.simulations, add_noise=False)
        distribution = root_policy(root, temperature=1.0)
        value = root.mean_value
    else:
        distribution, value = predict_position(board)
    legal_map = legal_action_map(board)
    ranked = sorted(distribution.items(), key=lambda x: x[1], reverse=True)[:req.top_k]
    payload = board_payload(board)
    payload.update({
        "value": float(value),
        "mode": "mcts" if req.simulations else "network",
        "moves": [{"uci": legal_map[a].uci(), "san": board.san(legal_map[a]),
                   "probability": p} for a, p in ranked],
    })
    return payload


@app.post("/api/play")
def play(req: PlayRequest):
    board = parse_board(req.fen)
    if not req.move:
        raise HTTPException(400, "A human move is required")
    try:
        human = chess.Move.from_uci(req.move)
    except ValueError:
        raise HTTPException(400, "Invalid UCI move")
    if human not in board.legal_moves:
        raise HTTPException(400, f"Illegal move: {req.move}")
    human_san = board.san(human)
    board.push(human)
    if board.is_game_over(claim_draw=True):
        payload = board_payload(board)
        payload["message"] = f"You played {human_san}. Game over: {payload['result']}"
        return payload
    with MODEL_LOCK:
        reply = model_move(board, req.simulations)
        reply_san = board.san(reply)
        board.push(reply)
    payload = board_payload(board)
    payload["message"] = f"You: {human_san} · Model: {reply_san}" + (
        f" · Result {payload['result']}" if payload["game_over"] else ""
    )
    return payload


@app.post("/api/model-move")
def make_model_move(req: PlayRequest):
    board = parse_board(req.fen)
    with MODEL_LOCK:
        move = model_move(board, req.simulations)
        san, uci = board.san(move), move.uci()
        board.push(move)
    payload = board_payload(board)
    payload.update({"move": uci, "message": f"Model played {san}"})
    return payload


def training_worker(req: TrainRequest):
    global JOB
    total = req.games + req.steps
    completed = 0
    try:
        with MODEL_LOCK:
            for i in range(req.games):
                record = play_self_game(req.simulations)
                completed += 1
                JOB = {"state": "running", "progress": completed / max(1, total),
                       "message": f"Self-play {i+1}/{req.games}: {record['result']} ({record['plies']} plies)"}
            for i in range(req.steps):
                metrics = train_one_step()
                completed += 1
                JOB = {"state": "running", "progress": completed / max(1, total),
                       "message": f"Training {i+1}/{req.steps}: loss {metrics['loss']:.4f}"}
        JOB = {"state": "done", "progress": 1.0,
               "message": f"Finished {req.games} game(s) and {req.steps} update(s)."}
    except Exception as exc:
        JOB = {"state": "error", "progress": completed / max(1, total), "message": str(exc)}
    finally:
        JOB_LOCK.release()


@app.post("/api/self-train")
def self_train(req: TrainRequest):
    if not JOB_LOCK.acquire(blocking=False):
        raise HTTPException(409, "A training job is already running")
    global JOB
    JOB = {"state": "running", "progress": 0.0, "message": "Starting self-play…"}
    threading.Thread(target=training_worker, args=(req,), daemon=True).start()
    return {"started": True}


@app.get("/api/weights")
def weights():
    if JOB.get("state") == "running":
        raise HTTPException(409, "Wait for the training job to finish")
    with MODEL_LOCK:
        return parameter_report()


@app.post("/api/weights/edit")
def edit_weight(req: WeightEdit):
    if JOB.get("state") == "running":
        raise HTTPException(409, "Wait for the training job to finish")
    if not math.isfinite(req.value) or abs(req.value) > 100:
        raise HTTPException(400, "Value must be finite and between -100 and 100")
    try:
        with MODEL_LOCK:
            old, new = set_weight(req.name, tuple(req.index), req.value)
        return {"old": old, "new": new}
    except (KeyError, ValueError, IndexError) as exc:
        raise HTTPException(400, str(exc))


@app.get("/api/checkpoint")
def checkpoint():
    with MODEL_LOCK:
        buffer = io.BytesIO()
        torch.save({"model": MODEL.state_dict(), "optimizer": OPTIMIZER.state_dict(),
                    "config": asdict(CFG), "train_history": TRAIN_HISTORY,
                    "game_history": GAME_HISTORY}, buffer)
    return Response(buffer.getvalue(), media_type="application/octet-stream",
                    headers={"Content-Disposition": "attachment; filename=chess_lab_checkpoint.pt"})


@app.post("/api/reset")
def reset():
    global MODEL, OPTIMIZER, SCALER, REPLAY, JOB
    if JOB.get("state") == "running":
        raise HTTPException(409, "Wait for the training job to finish")
    with MODEL_LOCK:
        seed_everything()
        MODEL = ChessNet().to(DEVICE)
        OPTIMIZER = torch.optim.AdamW(MODEL.parameters(), lr=CFG.learning_rate,
                                      weight_decay=CFG.weight_decay)
        SCALER = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
        REPLAY = ReplayBuffer(CFG.replay_capacity)
        TRAIN_HISTORY.clear(); GAME_HISTORY.clear()
        JOB = {"state": "idle", "progress": 0.0, "message": "Model reset."}
    return {"reset": True}


print("API routes:", [route.path for route in app.routes])


In [ ]:
#@title 13 · Start the server and open the lab
import nest_asyncio
nest_asyncio.apply()

PORT = 8000

# Stop a server created by an earlier execution of this cell.
if "SERVER" in globals() and SERVER is not None:
    SERVER.should_exit = True
    time.sleep(0.5)

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
SERVER = uvicorn.Server(config)
SERVER_THREAD = threading.Thread(target=SERVER.run, daemon=True)
SERVER_THREAD.start()
time.sleep(1.0)

try:
    from google.colab import output
    from IPython.display import HTML, display
    proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
    display(HTML(f"""<div style="padding:18px;border:1px solid #ccc;border-radius:12px">
      <b>Neural Chess Lab is running.</b><br><br>
      <a href="{proxy_url}" target="_blank" style="padding:10px 15px;background:#116466;color:white;border-radius:8px;text-decoration:none">Open browser lab ↗</a>
      <span style="margin-left:10px;color:#666">Keep this Colab runtime connected.</span>
    </div>"""))
except Exception:
    print(f"Open http://127.0.0.1:{PORT} in a local notebook environment.")


## 9 · Suggested student investigations

### Beginner

1. Compare the raw network (`0` simulations) and MCTS (`32` simulations) on the same position.
2. Generate one game, run 20 updates, and explain each telemetry metric.
3. Locate the policy head in the weight report. How is its shape related to 4,672 actions?

### Intermediate

1. Train two fresh models with different learning rates. Use the same seed and replay data.
2. Disable root noise and measure opening diversity across ten games.
3. Add horizontal board reflection as data augmentation. Remember to transform moves too.
4. Create an arena function that alternates colors and compares two checkpoints.

### Advanced

1. Add a transposition table keyed by Zobrist hash.
2. Batch leaf evaluations from multiple searches to use the GPU efficiently.
3. Gate new checkpoints: accept a candidate only if it beats the current champion in an arena.
4. Replace absolute planes with side-to-move canonicalization and test sample efficiency.
5. Add ELO estimation with confidence intervals rather than judging by training loss.

### Responsible interpretation

- A falling loss proves better fit to generated data, not necessarily stronger chess.
- Self-play can reinforce blind spots; evaluation must use held-out positions and opponent matches.
- Small classroom runs are noisy. Report seeds, compute budget, games, simulations, and confidence intervals.
- Never claim equivalence to AlphaZero based only on sharing its high-level loop.


In [ ]:
#@title 14 · Optional checkpoint utilities for Google Drive
# Uncomment to make checkpoints persist after the Colab runtime disconnects.
# from google.colab import drive
# drive.mount('/content/drive')
# save_checkpoint('/content/drive/MyDrive/chess_lab_checkpoint.pt')
# load_checkpoint('/content/drive/MyDrive/chess_lab_checkpoint.pt')
